# Week 9: Clustering Methods

We turn our attention to an "unsupervised" method of machine learning called clustering. "Unsupervised" refers to the fact that we do not have any pre-existing labels we are going to supply to the model in order to aide in the learning process. Therefore, the contrast is "supervised" learning in which we do provide labels for the model to learn boundaries. 

In clustering, our goal is to determine boundaries between groupings and place each data point within the grouping. There are a few main methods to go about clustering. 

In [ ]:
import pandas as pd 
import numpy as np 
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
import sklearn # sklearn has the long name of scikit-learn

In [ ]:
# We will read in data from the EPA on environmental justice

ej = pd.read_csv('EJI_2024_United_States.csv')
ej.head()

In [ ]:
# We'll keep only a few columns of interest right now
cols_of_interest = ["STATEABBR", # state abbreviation code
  "GEOID_2020", # census tract information - keep for plotting
  "E_POV200", # % below 200% poverty
  "E_NOHSDP", # % without HS diploma
  "E_UNEMP", # % unemployed
  "E_RENTER", # % renter-occupied homes
  "E_HOUBDN", # % housing cost burdened (make <$75k and pay >30% in housing costs)
  "E_MINRTY", # % minority population
  "E_LIMENG", # % limited English proficiency
  "E_PARK", # % of people within 1-mi of a green space
  "E_ROAD", # % of people within 1-mi of a high density road
  "E_DSLPM", # concentrations of diesel particulate matter
  "E_TOTCR"] # the percentile likelihood of developing cancer over ones lifetime 

ej_subset = ej.loc[:, cols_of_interest]
ej_subset.head()

In [ ]:
# Some of the "missing" data is input as -999
ej_subset = ej_subset.replace(-999, np.nan)

# Now we can remove rows that are missing key values
ej_subset = ej_subset.dropna()

# We also need the GEOID to be a string value
ej_subset['GEOID_2020'] = ej_subset['GEOID_2020'].astype(str)
ej_subset.describe()

In [ ]:
# Finally, let's focus on one state 
texas = ej_subset[ej_subset['STATEABBR'] == 'TX']

# And grab the geometry information from the census
# We need the geometries from the census 
year=2020
state_fips='48'
url = f"https://www2.census.gov/geo/tiger/TIGER{year}/TRACT/tl_{year}_{state_fips}_tract.zip"
tracts = gpd.read_file(url)

texas = pd.merge(left=texas, right=tracts, how='left', left_on='GEOID_2020', right_on='GEOID')
texas = gpd.GeoDataFrame(texas,
                         geometry='geometry', 
                         crs='EPSG:4326')


# And separate the GEO information for later
variables = cols_of_interest[2:]
texas_data = texas[variables]
texas_data.head()

In [ ]:
# We can visualize how these variables interact with each other
sns.pairplot(texas_data, kind="reg", diag_kind="kde")

## 1 Scaling

In this case, we know that many of these variables are percentages that are constrained to fall between zero and one, but some of the other variables like concentrations of particulate matter are not. A clustering algorithm that uses these values to determine classifications will pay a lot of attention to data with larger variations, but very little to the ones with small variations! 

Therefore, *as a rule*, we standardize our data when clustering. There are many different methods of standardization offered in the `sklearn.preprocessing` module, and these map onto the main methods common in applied work. We review a small subset of them here. The `scale()` method subtracts the mean and divides by the standard deviation:

$$ z = \frac{x_i - \bar{x}}{\sigma_x}$$

This "normalizes" the variate, ensuring the re-scaled variable has a mean of zero and a variance of one. However, the variable can still be quite skewed, bimodal, etc, and insofar as the mean and vairance may be affected by outliers in a given variate, the scaling can be too dramatic. One alternative intended to handle outliers better is `robust_scale()`, which uses the median and the interquartile range in the same fashion:

$$ z = \frac{x_i - \tilde{x}}{\lceil x \rceil_{75} - \lceil x \rceil_{25}}$$

where $\lceil x \rceil_p$ represents the value of the $p$th percentile of $x$. Alternatively, sometimes it is useful to ensure that the maximum of a variate is $1$ and the minimum is zero. In this instance, the `minmax_scale()` is appropriate: 

$$ z = \frac{x - min(x)}{max(x-min(x))} $$

In most clustering problems, the `robust_scale()` or `scale()` methods are useful. Further, transformations of the variate (such as log-transforming or Box-Cox transforms) can be used to nonlinearly rescale the variates, but these generally should be done before the above kinds of scaling. Here, we will analyze robust-scaled variables. To detach the scaling from the analysis, we will perform the former now, creating a scaled view of our data which we can use later for clustering. For this, we import the scaling method:

In [ ]:
from sklearn.preprocessing import robust_scale, scale, minmax_scale

## robust_scale is a function that scales the data to have a median of 0 and a
## standard deviation of 1. This is useful for clustering algorithms that are
## sensitive to the scale of the data.

In [ ]:
# We can scale our texas subset
texas_scaled = robust_scale(texas_data)
texas_scaled

## 2 K-means

K-means is probably the most widely used approach to cluster a dataset. The algorithm groups observations into a
pre-specified number of clusters so that that each observation is closer to the mean of its own cluster than it is to the mean of any other cluster. The k-means problem is solved by iterating between an assignment step and an update step.  First, all observations are randomly assigned one of the $k$ labels. Next, the  multivariate mean over all covariates is calculated for each of the clusters. Then, each observation is reassigned to the cluster with the closest mean.  If the observation is already assigned to the cluster whose mean it is closest to, the observation remains in that cluster. This assignment-update process continues until no further reassignments are necessary.

The nature of this algorithm requires us to select the number of clusters we  want to create. The right number of clusters is unknown in practice. For illustration, we will use $k=5$ in the `KMeans` implementation from `scikit-learn`. 

In [ ]:
from sklearn.cluster import KMeans

In [ ]:
# Initialise KMeans instance
kmeans = KMeans(n_clusters=5)

`KMeans` expects a matrix with the shape (n_rows, n_features). This is the format of our `texas_scaled`! If we wanted to just transform any pandas dataframe into this, we could call `df.values`

In [ ]:
# Set the seed for reproducibility
np.random.seed(0)
# Run K-Means algorithm
k5cls = kmeans.fit(texas_scaled)

In [ ]:
# Print first ten labels
k5cls.labels_[:10]

In this case, the first seven observations are assigned to cluster 1, the eighth to cluster 4 and tenth is assigned to cluster 0. It is important to note that the integer labels should be viewed as denoting membership only &mdash;
the numerical differences between the values for the labels are meaningless. The profiles of the various clusters must be further explored by looking at the values of each dimension. But, before we do that, let's make a map.


Having obtained the cluster labels, we can display the spatial
distribution of the clusters by using the labels as the categories in a
choropleth map. This allows us to quickly grasp any sort of spatial pattern the 
clusters might have. Since clusters represent areas with similar
characteristics, mapping their labels allows to see to what extent similar areas tend
to have similar locations.
Thus, this gives us one map that incorporates the information from all eleven covariates.

In [ ]:
# Assign labels into a column, back to our full texas data with the geometry column
texas["k5cls"] = k5cls.labels_



In [ ]:
# Setup figure and ax
f, ax = plt.subplots(1, figsize=(9, 9))
# Plot unique values choropleth including
# a legend and with no boundary lines
texas.plot(column="k5cls", categorical=True, legend=True, linewidth=0, ax=ax)
# Remove axis
ax.set_axis_off()


How do we interpret the clusters though? What do they represent from our data? We can `groupby` each cluster label and invesigate the numerical properties of each of the covariates. 

In [ ]:
# Let's use our non-scaled data so we understand the values

texas.groupby('k5cls')[variables].mean()

We are looking at the means to show us the cluster "centers" of each variable. We can summarize along clusters to describe them. We can also look along variables to see if there is a big discrepancy between clusters. This means the feature has high importance in separating them. 

So what do we see?

* Cluster 0: high % poverty, renters, household burden, very high minority population living within 1-mi of a park or a road
* Cluster 1: low % without HS diploma, unemployed, with limited english. More likely living far from a highway or park, low concentrations of diesel.
* Cluster 2: low % povery,without HS diploma, renters, household burden, limited english. Lives close to parks and highways, higher rates of diesel matter and cancer risk. 
* Cluster 3: higher % povery, but lower % without HS diploma or unemployed. High % renters and household burden. Lives close to parks and highways, higher rates of diesel matter and cancer risk. 
* Cluster 4: high % pverty, no HS diploma, renters, household burden, minority population. Higher % with limited english. Lives close to parks and highways. Higher rates of diesel matter and cancer risk. 

In [ ]:
## YOUR TURN 
## Pick a different k value and plot the map, what changes? 



In [ ]:
## YOUR TURN
## Compute the means of all the variables in each cluster, what do we see?



# 3 DBSCAN 

From the point of view of **DBSCAN, a cluster is a concentration of at least `minPts` points, each of them within a distance of `epsilon` of at least another point in the cluster**. Following this definition, the algorithm classifies each point in our pattern into three categories:

* *Noise*, for those points outside a cluster.
* *Cores*, for those points inside a cluster with at least `m` points in the cluster within distance `r`.
* *Borders* for points inside a cluster with less than `m` other points in the cluster within distance `r`.

Here is a cluster where `m` is 3 points for a given `r` distance.

</figure>
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/a/af/DBSCAN-Illustration.svg/400px-DBSCAN-Illustration.svg.png" alt="drawing" width="600" style="display: block; margin: 0 auto"/>
</figure>

The flexibility (but also some of the limitations) of the algorithm resides in that both `m` and `r` need to be specified by the user before running DBSCAN. This is a critical point, as their value can influence significantly the final result. Before exploring this in greater depth, let us get a first run at computing `DBSCAN` in Python:


In [ ]:
from sklearn.cluster import DBSCAN

In [ ]:
# Define DBSCAN
db = DBSCAN(eps=1, min_samples=5)

In [ ]:
# Just as with k-means
# Set the seed for reproducibility
np.random.seed(0)
# Run DBSCAN algorithm
db_clusters = db.fit(texas_scaled)

In [ ]:
# As before we can see the labels
db_clusters.labels_[:10]

-1 represents noise labels. 

In [ ]:
# Plot spatially
texas['dbcls'] = db_clusters.labels_
# Setup figure and ax
f, ax = plt.subplots(1, figsize=(9, 9))
# Plot unique values choropleth including
# a legend and with no boundary lines
texas.plot(column="dbcls", categorical=True, legend=True, linewidth=0, ax=ax)
# Remove axis
ax.set_axis_off()

In [ ]:
# And we can interpret the same way
# Let's use our non-scaled data so we understand the values
texas.groupby('dbcls')[variables].mean()

What do we see? 

In [ ]:
## YOUR TURN 
## Change minPts and epsilon to see how the clusters change in comparison to our reference plot


## 4 Exploration

If time permits, choose a new set of variables or a new state and run one of the clustering algorithms